In [82]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import os

In [83]:
paths = ["raw//"+ x for x in os.listdir("raw") if x.endswith('.csv')]

In [84]:
import os

print(os.getcwd())
for p in paths:
    print(p, "->", os.path.exists(p))


c:\Users\mohak\OneDrive\Documents\DATASET\archive
raw//Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv -> True
raw//Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv -> True
raw//Friday-WorkingHours-Morning.pcap_ISCX.csv -> True
raw//Monday-WorkingHours.pcap_ISCX.csv -> True
raw//Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv -> True
raw//Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv -> True
raw//Tuesday-WorkingHours.pcap_ISCX.csv -> True
raw//Wednesday-workingHours.pcap_ISCX.csv -> True


In [85]:
paths


['raw//Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv',
 'raw//Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv',
 'raw//Friday-WorkingHours-Morning.pcap_ISCX.csv',
 'raw//Monday-WorkingHours.pcap_ISCX.csv',
 'raw//Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv',
 'raw//Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv',
 'raw//Tuesday-WorkingHours.pcap_ISCX.csv',
 'raw//Wednesday-workingHours.pcap_ISCX.csv']

In [86]:
test_path = paths[:3]
train_path = ['raw//Monday-WorkingHours.pcap_ISCX.csv',
             'raw//Tuesday-WorkingHours.pcap_ISCX.csv',
             'raw//Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv',
             'raw//Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv',
             'raw//Wednesday-workingHours.pcap_ISCX.csv']
test_path = test_path[::-1]
test_path

['raw//Friday-WorkingHours-Morning.pcap_ISCX.csv',
 'raw//Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv',
 'raw//Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv']

In [87]:
columns_list = [pd.read_csv(p,nrows=0).columns.to_list() for p in paths]
  
all_same = all(cols == columns_list[0] for cols in columns_list)
print("All datasets have matching columns:", all_same)

All datasets have matching columns: True


In [88]:
dfs=[pd.read_csv(p) for p in train_path]

merged_df_train = pd.concat(dfs,axis=0,ignore_index=True) 


dfs=[pd.read_csv(p) for p in test_path]

merged_df_test = pd.concat(dfs,axis=0,ignore_index=True) 
print(len(merged_df_train)+len(merged_df_test))

#total should be 2830743 rows

2830743


In [89]:
merged_df_test.columns

Index([' Destination Port', ' Flow Duration', ' Total Fwd Packets',
       ' Total Backward Packets', 'Total Length of Fwd Packets',
       ' Total Length of Bwd Packets', ' Fwd Packet Length Max',
       ' Fwd Packet Length Min', ' Fwd Packet Length Mean',
       ' Fwd Packet Length Std', 'Bwd Packet Length Max',
       ' Bwd Packet Length Min', ' Bwd Packet Length Mean',
       ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s',
       ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min',
       'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max',
       ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std',
       ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags',
       ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length',
       ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s',
       ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean',
       ' Packet Length Std', ' Packet Length Variance', '

In [90]:
merged_df_train[' Label'].value_counts()

 Label
BENIGN                        1858775
DoS Hulk                       231073
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64

In [91]:
merged_df_test[' Label'].value_counts()

 Label
BENIGN      414322
PortScan    158930
DDoS        128027
Bot           1966
Name: count, dtype: int64

In [92]:
bot_df = merged_df_test[merged_df_test[" Label"]=="Bot"]
ddos_df = merged_df_test[merged_df_test[" Label"]=="DDoS"]
port_df = merged_df_test[merged_df_test[" Label"]=="PortScan"]
benign_df = merged_df_test[merged_df_test[" Label"]=="BENIGN"]

BOT_TRAIN_FRAC = 0.50
DDOS_TRAIN_FRAC = 0.20
PORT_TRAIN_FRAC = 0.20
RANDOM_SEED = 42

bot_train = bot_df.sample(frac=BOT_TRAIN_FRAC,random_state=RANDOM_SEED)
port_train = port_df.sample(frac=PORT_TRAIN_FRAC,random_state=RANDOM_SEED)
ddos_train = ddos_df.sample(frac=DDOS_TRAIN_FRAC,random_state=RANDOM_SEED)

bot_test = bot_df.drop(bot_train.index)
port_test = port_df.drop(port_train.index)
ddos_test = ddos_df.drop(ddos_train.index)

test_df_final = pd.concat([bot_test,port_test,ddos_test,benign_df],axis=0)
merged_df_train = pd.concat([merged_df_train,bot_train,port_train,ddos_train],axis=0)


In [93]:
merged_df_train[' Label'].value_counts()

 Label
BENIGN                        1858775
DoS Hulk                       231073
PortScan                        31786
DDoS                            25605
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Web Attack � Brute Force         1507
Bot                               983
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64

In [94]:
import re

def clean_web_attack_labels(df):
    
    # 1. Clean 'Web Attack - Brute Force'
    df[' Label'] = df[' Label'].str.replace(
        r'^Web Attack.*Brute Force$', 
        'Web Attack - Brute Force', 
        regex=True
    )

    # 2. Clean 'Web Attack - XSS'
    df[' Label'] = df[' Label'].str.replace(
        r'^Web Attack.*XSS$', 
        'Web Attack - XSS', 
        regex=True
    )
    
    # 3. Clean 'Web Attack - Sql Injection'
    df[' Label'] = df[' Label'].str.replace(
        r'^Web Attack.*Sql Injection$', 
        'Web Attack - Sql Injection', 
        regex=True
    )
    
    return df

merged_df_train = clean_web_attack_labels(merged_df_train.copy())

In [95]:
merged_df_train[' Label'].value_counts() 

 Label
BENIGN                        1858775
DoS Hulk                       231073
PortScan                        31786
DDoS                            25605
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Web Attack - Brute Force         1507
Bot                               983
Web Attack - XSS                  652
Infiltration                       36
Web Attack - Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64

In [96]:
SMALL_ATTACKS = [
    'Infiltration', 
    'Web Attack - Sql Injection', 
    'Heartbleed',
    'Web Attack - XSS'
]

NEW_LABEL = "Other Attack"
merged_df_train[' Label'] = merged_df_train[' Label'].replace(SMALL_ATTACKS, NEW_LABEL)

In [97]:
merged_df_train[' Label'].value_counts() 

 Label
BENIGN                      1858775
DoS Hulk                     231073
PortScan                      31786
DDoS                          25605
DoS GoldenEye                 10293
FTP-Patator                    7938
SSH-Patator                    5897
DoS slowloris                  5796
DoS Slowhttptest               5499
Web Attack - Brute Force       1507
Bot                             983
Other Attack                    720
Name: count, dtype: int64

In [98]:
TARGET_BENIGN_COUNT = 30000

benign_mask = (merged_df_train[' Label']=="BENIGN")


benign_train_undersampled = merged_df_train[benign_mask].sample(
    n = TARGET_BENIGN_COUNT,
    random_state=RANDOM_SEED
)
print(f"Sampled BENIGN count: {len(benign_train_undersampled):,}")

attack_mask = (merged_df_train[' Label'] != 'BENIGN')
attack_train_df = merged_df_train[attack_mask]
print(f"Total Attack count preserved: {len(attack_train_df):,}")


train_df_final = pd.concat(
    [attack_train_df,benign_train_undersampled],
    axis=0
)

Sampled BENIGN count: 30,000
Total Attack count preserved: 327,097


In [ ]:
#for test df
TARGET_BENIGN_COUNT = 120000

benign_mask = (test_df_final[' Label']=="BENIGN")


benign_train_undersampled = test_df_final[benign_mask].sample(
    n = TARGET_BENIGN_COUNT,
    random_state=RANDOM_SEED
)
print(f"Sampled BENIGN count: {len(benign_train_undersampled):,}")

attack_mask = (test_df_final[' Label'] != 'BENIGN')
attack_train_df = test_df_final[attack_mask]
print(f"Total Attack count preserved: {len(attack_train_df):,}")


test_df_final = pd.concat(
    [attack_train_df,benign_train_undersampled],
    axis=0
)

Sampled BENIGN count: 120,000
Total Attack count preserved: 230,549


In [100]:
TARGET_DOS_COUNT = 30000

dos_mask = (train_df_final[' Label']=="DoS Hulk")


dos_train_undersampled = train_df_final[dos_mask].sample(
    n = TARGET_DOS_COUNT,
    random_state=RANDOM_SEED
)
print(f"Sampled Dos Hulk count: {len(dos_train_undersampled):,}")

attack_mask = (train_df_final[' Label'] != 'DoS Hulk')
attack_train_df = train_df_final[attack_mask]
print(f"Total Attack count preserved: {len(attack_train_df):,}")


train_df_final = pd.concat(
    [attack_train_df,dos_train_undersampled],
    axis=0
)

Sampled Dos Hulk count: 30,000
Total Attack count preserved: 126,024


In [101]:

print(f"\nFinal balanced training set size: {len(train_df_final):,}")


Final balanced training set size: 156,024


In [102]:
train_df_final[' Label'].value_counts() 

 Label
PortScan                    31786
BENIGN                      30000
DoS Hulk                    30000
DDoS                        25605
DoS GoldenEye               10293
FTP-Patator                  7938
SSH-Patator                  5897
DoS slowloris                5796
DoS Slowhttptest             5499
Web Attack - Brute Force     1507
Bot                           983
Other Attack                  720
Name: count, dtype: int64

In [103]:
test_df_final[' Label'].value_counts() 

 Label
PortScan    127144
BENIGN      120000
DDoS        102422
Bot            983
Name: count, dtype: int64

In [104]:
train_df_final.to_parquet('train_final.parquet')
test_df_final.to_parquet('test_final.parquet')